# Random walks and diffusion in Julia

I begin with molecules at the origin on an unbounded two-dimensional lattice. Each step moves one lattice spacing in a cardinal direction. Position is measured in lattice spacings and time in sweeps. One sweep moves every molecule once. Mean-square displacement is the average squared distance from the starting point, in squared lattice spacings. The diffusion coefficient for this convention is 1/4 lattice-spacing squared per sweep.

Run each small part and its checks in order. Julia indices start at 1. For Fourier examples install FFTW with `julia -e 'using Pkg; Pkg.add("FFTW")'`. For notebook use, install IJulia and select its kernel. The script is also supplied. Julia and NumPy random sequences differ; compare ensemble statistics, not individual paths.

**Using my code.** I permit free noncommercial teaching, learning and demonstrations with acknowledgment. Research (including academic research) and commercial use require my explicit written permission. Earlier MIT and GPL releases retain their existing permissions. Read the [code-use terms](https://saswataiith.github.io/files/CODE-USE-TERMS.txt). Request permission at saswata_at_msme.iith.ac.in. These terms apply to code; teaching text and figures have separate terms.


## 1. Set the positions and allowed moves

I first check the move lengths. No boundary rule is applied to the positions: the histogram window used by the complete program is only a display window.

In [ ]:
using Random, Statistics
rng = MersenneTwister(7)
number = 50000
positions = zeros(Int, number, 2)
moves = [1 0; -1 0; 0 1; 0 -1]
@assert all(sum(moves.^2; dims=2) .== 1)
@assert all(positions .== 0)
println("Molecules: ", number)

## 2. Take one sweep

I choose a direction independently for each molecule. After the first sweep every molecule is exactly one lattice spacing from the origin.

In [ ]:
for molecule in axes(positions, 1)
    choice = rand(rng, 1:4)
    positions[molecule, :] .+= moves[choice, :]
end
@assert all(sum(positions.^2; dims=2) .== 1)
println("First-sweep mean-square displacement: ", mean(sum(positions.^2; dims=2)))

## 3. Measure displacement over many sweeps

The predicted ensemble mean-square displacement equals the sweep count. An individual finite sample fluctuates. I therefore use a stated statistical tolerance instead of requiring exact equality. The seed makes this NumPy run repeatable; Julia uses a different random generator.

In [ ]:
sweep_count = 100
history = [0.0, 1.0]
for sweep in 2:sweep_count
    for molecule in axes(positions, 1)
        choice = rand(rng, 1:4)
        positions[molecule, :] .+= moves[choice, :]
    end
    push!(history, mean(sum(positions.^2; dims=2)))
end
relative_error = abs(history[end] / sweep_count - 1)
@assert relative_error < 0.06
println("Measured: ", history[end], "; prediction: ", sweep_count)

## 4. Separate individual motion from spreading

A molecule continues making discrete random steps. Diffusion describes the distribution of many molecules. The mean displacement should be small compared with the root-mean-square distance. The check below is an ensemble check for this seeded example, not a universal bound.

In [ ]:
mean_position = vec(mean(positions; dims=1))
root_mean_square_distance = sqrt(history[end])
@assert sqrt(sum(mean_position.^2)) < 0.1 * root_mean_square_distance
println("Mean position: ", mean_position)
println("Root-mean-square distance: ", root_mean_square_distance)